In [1]:
from subprocess import run
import pandas as pd
import os
from nilearn import datasets,image, surface, plotting
import nibabel as nib
import numpy as np
import plotly.io as pio  # Import plotly.io for exporting images
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import warnings


def get_subbricks_info(dataset):
    # Run 3dinfo command
    result = run(['/work/apps/AFNI/linux_openmp_64/3dinfo', '-verb', dataset], capture_output=True, text=True)
    lines = result.stdout.split('\n')
    
    # Extract lines with sub-brick info
    subbrick_lines = [line for line in lines if "sub-brick" in line]
    
    # Extract indices and names from those lines
    subbricks = []
    for line in subbrick_lines:
        idx = int(line.split('#')[1].split()[0])
        name = line.split("'")[1]
        subbricks.append((idx, name))
        
    return subbricks
Dir_github = '/work/desai-lab/xuanyang/Project/Semantic/analysis/FactorAnalysis/github/FactorAnalysis_fMRI/scripts/'
Dir_mask = os.path.join(Dir_github,'fMRI','masks')


/work/svs-lab/xuanyang/ENVS/hyperalignment/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


ModuleNotFoundError: No module named 'plotly'

# Skip if already run

In [1]:
# Skip if already run

file_master = '/work/desai-lab/xuanyang/Project/Semantic/analysis/ParametricModulation/Nastase/allstories/FactorAnalysis/models/Nvar113NFA7_LPAC/FA7/results/firstlevel/sub-023/milkyway/sub-023.results/stats.sub-023_REML+tlrc.BRIK'
file_input = '/work/xy6/templates/HCPex_v1.1/HCPex.nii.gz'
file_output = 'rs_HCPex.nii.gz'

os.chdir(Dir_mask)

run(f"/work/apps/AFNI/linux_openmp_64/3dresample -master {file_master} -input {file_input} -prefix {file_output}",shell=True)


CompletedProcess(args='/work/apps/AFNI/linux_openmp_64/3dresample -master /work/desai-lab/xuanyang/Project/Semantic/analysis/ParametricModulation/Nastase/allstories/FactorAnalysis/models/Nvar113NFA7_LPAC/FA7/results/firstlevel/sub-023/milkyway/sub-023.results/stats.sub-023_REML+tlrc.BRIK -input /work/xy6/templates/HCPex_v1.1/HCPex.nii.gz -prefix rs_HCPex.nii.gz', returncode=0)

# Get ROI values

In [127]:
mastersheetFile = '/work/desai-lab/xuanyang/Project/Semantic/dissemination/github/DiscoFMRI/scripts/fMRI/master_subject_10stories_highacc.csv'
df_master = pd.read_csv(mastersheetFile)
df_master.loc[df_master['transcript'].isin(['prettymouth','milkywayoriginal','milkywayvodka','slumlordreach','21styear']),'protocol'] = 'skyra'
df_master.loc[df_master['transcript'].isin(['shapessocial']),'protocol'] = 'Prisma_MB4'
df_master.loc[df_master['transcript'].isin(['piemanpni','bronx','black','forgot']),'protocol'] = 'Prisma_MB3'


Nvar = 113
NFA = 8
flag_model = f'Nvar{Nvar}NFA{NFA}_LPAC_multipleReg_unsmooth'
Dir_working = '/work/desai-lab/xuanyang/Project/Semantic/analysis/ParametricModulation/Nastase/allstories/FactorAnalysis'
Dir_model = os.path.join(Dir_working,'models',flag_model,f'FA{NFA}')
Dir_results = os.path.join(Dir_model,'results')
Dir_results_1st = os.path.join(Dir_results,'firstlevel')
Dir_results_2nd = os.path.join(Dir_results,'secondlevel')

for i,subID, task,label in zip(df_master.index,df_master.subID,df_master.task,df_master.label):
    idx = "{:03d}".format(i+1)
    Dir_sub_task = os.path.join(Dir_results_1st,subID,task)
    path_img_stats = os.path.join(Dir_sub_task,'{}.results'.format(subID),'stats.{}_REML+tlrc.BRIK'.format(subID))
    df_master.loc[i,'finished'] = 0

    if os.path.exists(path_img_stats):
        df_master.loc[i,'finished'] = 1
        df_master.loc[i,'img_stats'] = path_img_stats

subfolder = '{}highacc'.format(len(df_master))
subList_ana_cmplt = df_master.loc[df_master.finished==1].reset_index(drop=True)
Nsub = len(df_master)

df_subbricks = pd.DataFrame(get_subbricks_info(subList_ana_cmplt.img_stats[0]),columns=['imgIDList', 'regName'])
df_regModel = df_subbricks.loc[df_subbricks.regName.str.contains(f'_bin2345#0_Coef')].copy()
df_regModel['regName'] = df_regModel['regName'].str.replace('_bin2345#0_Coef',f'_{flag_model}')
df_regModel['regName'] = df_regModel['regName'] +'_'+subfolder


In [27]:
file_atlas = os.path.join(Dir_mask,'rs_HCPex.nii.gz')
img_atlas = image.load_img(file_atlas)
data_atlas = img_atlas.get_fdata()

file_mask = '/work/desai-lab/xuanyang/Project/Semantic/analysis/FactorAnalysis/github/FactorAnalysis_fMRI/scripts/fMRI/masks/rs_mask_GM_33.nii.gz'
img_mask = image.load_img(file_mask)
data_mask = img_mask.get_fdata()

data_atlas[data_mask==0] = 0

In [147]:
atlas_name = 'HCPex'
Dir_output = os.path.join(Dir_working,'models',flag_model,f"FA{NFA}_{atlas_name}",'results','firstlevel')
if not os.path.exists(Dir_output):
    os.makedirs(Dir_output)
    
for i,row in df_master.iterrows():
        
    for iFA,regName,imgID in zip(range(1,NFA+1),df_regModel['regName'],df_regModel['imgIDList']): # FAs
        
        file_FA = row['img_stats']
        
        if '.nii' in file_FA:
            img_FA = image.load_img(file_FA)
        else:
            img_FA = image.index_img(file_FA, imgID) 
        
        data_FA = img_FA.get_fdata()
        
        data_output = np.zeros(data_FA.shape)
        
        for j,roi in enumerate(np.unique(data_atlas)):
            data_output[(data_atlas==roi)&(data_mask==1)] = data_FA[(data_atlas==roi)&(data_mask==1)].mean()
            
        # export
        Dir_sub = os.path.join(Dir_output,row['subID'],row['task'])
        if not os.path.exists(Dir_sub):
            os.makedirs(Dir_sub)
    
        new_img = image.new_img_like(img_mask, data_output)
        output_path = os.path.join(Dir_sub, f"{row['subID']}_{flag_model}_{atlas_name}_FA{iFA}.nii.gz")
        new_img.to_filename(output_path)
    print(i,row['subID'])

0 sub-023
1 sub-023
2 sub-030
3 sub-030
4 sub-032
5 sub-032
6 sub-034
7 sub-034
8 sub-049
9 sub-050
10 sub-052
11 sub-052
12 sub-058
13 sub-065
14 sub-066
15 sub-075
16 sub-079
17 sub-079
18 sub-081
19 sub-081
20 sub-083
21 sub-083
22 sub-084
23 sub-086
24 sub-086
25 sub-087
26 sub-087
27 sub-088
28 sub-088
29 sub-089
30 sub-089
31 sub-090
32 sub-090
33 sub-091
34 sub-091
35 sub-093
36 sub-093
37 sub-095
38 sub-095
39 sub-096
40 sub-096
41 sub-097
42 sub-097
43 sub-098
44 sub-098
45 sub-099
46 sub-099
47 sub-100
48 sub-100
49 sub-101
50 sub-101
51 sub-102
52 sub-102
53 sub-103
54 sub-103
55 sub-104
56 sub-104
57 sub-106
58 sub-106
59 sub-107
60 sub-107
61 sub-108
62 sub-108
63 sub-109
64 sub-109
65 sub-110
66 sub-110
67 sub-111
68 sub-111
69 sub-127
70 sub-127
71 sub-131
72 sub-136
73 sub-143
74 sub-145
75 sub-186
76 sub-190
77 sub-190
78 sub-201
79 sub-221
80 sub-222
81 sub-223
82 sub-225
83 sub-228
84 sub-229
85 sub-230
86 sub-231
87 sub-232
88 sub-233
89 sub-234
90 sub-235
91 sub-23